In [27]:
import numpy as np
import pandas as pd
import os
import json
import joblib
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,average_precision_score,confusion_matrix,f1_score,precision_recall_curve
from datetime import datetime
print("imports successful..")
print("xgb version",xgb.__version__)

imports successful..
xgb version 3.2.0


In [28]:
fraud_Data_path=(r"C:\Users\vishw\Desktop\projects\mini projects aiml\credit_card\creditcard.csv")
fraud_Data=pd.read_csv(fraud_Data_path)
fraud_Data.shape

(284807, 31)

In [29]:
fraud_Data.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [30]:
fraud_Data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

In [31]:
fraud_Data.describe()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,284807.000000,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,...,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,284807.000000,284807.000000
mean,94813.859575,1.175161e-15,3.384974e-16,-1.379537e-15,2.094852e-15,1.021879e-15,1.494498e-15,-5.620335e-16,1.149614e-16,-2.414189e-15,...,1.628620e-16,-3.576577e-16,2.618565e-16,4.473914e-15,5.109395e-16,1.686100e-15,-3.661401e-16,-1.227452e-16,88.349619,0.001727
std,47488.145955,1.958696e+00,1.651309e+00,1.516255e+00,1.415869e+00,1.380247e+00,1.332271e+00,1.237094e+00,1.194353e+00,1.098632e+00,...,7.345240e-01,7.257016e-01,6.244603e-01,6.056471e-01,5.212781e-01,4.822270e-01,4.036325e-01,3.300833e-01,250.120109,0.041527
min,0.000000,-5.640751e+01,-7.271573e+01,-4.832559e+01,-5.683171e+00,-1.137433e+02,-2.616051e+01,-4.355724e+01,-7.321672e+01,-1.343407e+01,...,-3.483038e+01,-1.093314e+01,-4.480774e+01,-2.836627e+00,-1.029540e+01,-2.604551e+00,-2.256568e+01,-1.543008e+01,0.000000,0.000000
25%,54201.500000,-9.203734e-01,-5.985499e-01,-8.903648e-01,-8.486401e-01,-6.915971e-01,-7.682956e-01,-5.540759e-01,-2.086297e-01,-6.430976e-01,...,-2.283949e-01,-5.423504e-01,-1.618463e-01,-3.545861e-01,-3.171451e-01,-3.269839e-01,-7.083953e-02,-5.295979e-02,5.600000,0.000000
50%,84692.000000,1.810880e-02,6.548556e-02,1.798463e-01,-1.984653e-02,-5.433583e-02,-2.741871e-01,4.010308e-02,2.235804e-02,-5.142873e-02,...,-2.945017e-02,6.781943e-03,-1.119293e-02,4.097606e-02,1.659350e-02,-5.213911e-02,1.342146e-03,1.124383e-02,22.000000,0.000000
75%,139320.500000,1.315642e+00,8.037239e-01,1.027196e+00,7.433413e-01,6.119264e-01,3.985649e-01,5.704361e-01,3.273459e-01,5.971390e-01,...,1.863772e-01,5.285536e-01,1.476421e-01,4.395266e-01,3.507156e-01,2.409522e-01,9.104512e-02,7.827995e-02,77.165000,0.000000
max,172792.000000,2.454930e+00,2.205773e+01,9.382558e+00,1.687534e+01,3.480167e+01,7.330163e+01,1.205895e+02,2.000721e+01,1.559499e+01,...,2.720284e+01,1.050309e+01,2.252841e+01,4.584549e+00,7.519589e+00,3.517346e+00,3.161220e+01,3.384781e+01,25691.160000,1.000000


In [32]:
#DATA PREPROCESSING 
#dropping unnamed column and removing duplicate values
if "Unnamed :0" in fraud_Data.columns:
    fraud_Data.drop(columns="Unnamed:0",axis=1,inplace=True)
initial_rows=fraud_Data.shape[0]  
subset=[c for c in fraud_Data.columns if c not in ["Time","Class"]]  
fraud_Data.drop_duplicates(subset=subset, inplace=True)  
print(f"🧹 Removed {initial_rows - fraud_Data.shape[0]} duplicate transactions")
print(f"✅ Clean shape: {fraud_Data.shape}")

🧹 Removed 9144 duplicate transactions
✅ Clean shape: (275663, 31)


# what if the column "Unnamed" is saved as "unnamed" in the dataset 

1. catch all variants of the Unnamed in the dataset[ UNNamed, unnamed]
phantom_cols = [col for col in data.columns if 'unnamed' in str(col).lower()]

2. If the list is not empty, drop them all at once
if phantom_cols:
    data.drop(phantom_cols, axis=1, inplace=True)
    print(f"🧹 Dropped phantom columns: {phantom_cols}")

In [ ]:
#FEATURE ENGINEERING (new features : AMOUNT BINS )
print("1. HOUR OF DAY:")
#converting Time column into hour of day to let the model know about the exact time of transaction
#converts the seconds elapsed into 24 hour clock
fraud_Data["Hour_of_Day"]=(fraud_Data['Time']//3600) %24
print("feature 1 created succesfully...")
print("2.IS_NIGHT:")
#creates a time window to check whether the fraud occurs between 11 pm to 6am
fraud_Data["IS_NIGHT"]=fraud_Data["Hour_of_Day"].apply(lambda x:1 if x<=6 or x>=23 else 0) 
print("feature 2 created successfully...")
print("3:LOG AMOUNT:")
#uses a log (log1p) to squash large values and minimize the difference between large values and smaller values
fraud_Data["LOG_AMOUNT"]=np.log1p(fraud_Data['Amount'])
print("feature 3 created successfully...")
#new feature
print("4. AMOUNT_SIZE:")
# NEW Feature 4: Amount bins — categorical signal for transaction size tier
# Fraudsters often test with micro-transactions before a big hit
fraud_Data['AMOUNT_TIER'] = pd.cut(
    fraud_Data['Amount'],
    bins=[-1, 1, 10, 100, 1000, float('inf')],
    labels=[0, 1, 2, 3, 4]  # micro, small, medium, large, whale
).astype(int)
print("✅ Feature 4: Amount_tier created (micro/small/medium/large/whale)")
# Dropping Time column
fraud_Data.drop(['Time','Amount'], axis=1, inplace=True)


1. HOUR OF DAY:
feature 1 created succesfully...
2.IS_NIGHT:
feature 2 created successfully...
3:LOG AMOUNT:
feature 3 created successfully...
4. AMOUNT_SIZE:
✅ Feature 4: Amount_tier created (micro/small/medium/large/whale)


In [34]:
fraud_Data.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V24,V25,V26,V27,V28,Class,Hour_of_Day,IS_NIGHT,LOG_AMOUNT,AMOUNT_TIER
0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,...,0.066928,0.128539,-0.189115,0.133558,-0.021053,0,0.0,1,5.014760,3
1,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,...,-0.339846,0.167170,0.125895,-0.008983,0.014724,0,0.0,1,1.305626,1
2,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,...,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,0,0.0,1,5.939276,3
3,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,...,-1.175575,0.647376,-0.221929,0.062723,0.061458,0,0.0,1,4.824306,3
4,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,...,0.141267,-0.206010,0.502292,0.219422,0.215153,0,0.0,1,4.262539,2


In [35]:
#SEPARATING FEATURES AND TARGET 
data_x=fraud_Data.drop("Class",axis=1)
data_y=fraud_Data["Class"]
data_y

0         0
1         0
2         0
3         0
4         0
         ..
284802    0
284803    0
284804    0
284805    0
284806    0
Name: Class, Length: 275663, dtype: int64

In [36]:
data_x


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V23,V24,V25,V26,V27,V28,Hour_of_Day,IS_NIGHT,LOG_AMOUNT,AMOUNT_TIER
0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,...,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,0.0,1,5.014760,3
1,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,...,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,0.0,1,1.305626,1
2,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,...,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,0.0,1,5.939276,3
3,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,...,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,0.0,1,4.824306,3
4,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,...,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,0.0,1,4.262539,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,4.356170,...,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,23.0,1,0.570980,0
284803,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,-0.975926,...,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,23.0,1,3.249987,2
284804,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,-0.484782,...,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,23.0,1,4.232366,2
284805,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,-0.399126,...,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,23.0,1,2.397895,1


In [37]:
#SPLITTING DATA
data_x_training,data_x_testing,data_y_training,data_y_testing=train_test_split(data_x,data_y,test_size=0.2,random_state=42,stratify=data_y,)
print(f"Train size: {data_x_training.shape[0]} | Test size: {data_x_testing.shape[0]}")


Train size: 220530 | Test size: 55133


In [38]:
# ── CELL 6: Save ROI reference BEFORE scaling ──────────────────────────────

# Re-read original Amount for test-set ROI reporting
# We do this from the original CSV to avoid confusion with the engineered features
raw_data_for_roi = pd.read_csv(fraud_Data_path)
subset_roi = [c for c in raw_data_for_roi.columns if c not in ['Time', 'Class']]
raw_data_for_roi.drop_duplicates(subset=subset_roi, inplace=True)

# Use same random_state=42 split to get matching test-set amounts
_, amount_test_series, _, _ = train_test_split(
    raw_data_for_roi['Amount'], raw_data_for_roi['Class'],
    test_size=0.2, random_state=42, stratify=raw_data_for_roi['Class']
)
amount_test = amount_test_series.values

print(f"✅ ROI reference saved: {len(amount_test)} test transactions")
print(f"   Total $ at stake in test set: ${amount_test[data_y_testing==1].sum():,.2f}")

✅ ROI reference saved: 55133 test transactions
   Total $ at stake in test set: $13,581.38


In [39]:
#APPLYING STANDARD SCALER
scaler=StandardScaler()
data_x_training_scaled=pd.DataFrame(scaler.fit_transform(data_x_training),columns=data_x_training.columns)
data_x_testing_scaled=pd.DataFrame(scaler.transform(data_x_testing),columns=data_x_testing.columns)
print(f"✅ Scaling done. Features: {list(data_x_training_scaled.columns)}")

✅ Scaling done. Features: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Hour_of_Day', 'IS_NIGHT', 'LOG_AMOUNT', 'AMOUNT_TIER']


In [40]:
#MODEL TRAINING (V3 VERSION)

#calculating the fraud ratio
fraud_Weight=len(data_y_training[data_y_training==0]) / len(data_y_training[data_y_training==1])
print(f"Class imbalance ratio (fraud_weight): {fraud_Weight:.2f}")
print("(Each fraud sample treated as if it were this many normal samples)\n")

model_parms={
    "n_estimators":600,
    "max_depth":6,
    "random_state":42,
    "learning_rate":0.05,
    "scale_pos_weight": fraud_Weight,
    "n_jobs":-1,
    "eval_metric":"aucpr",       # V3 FIX: metric now actually used with eval_set
    "early_stopping_rounds":50,   # V3 NEW: stops if no improvement after 20 rounds
    'subsample': 0.8,            # V3 NEW: row sampling per tree (reduces overfitting)
    'colsample_bytree': 0.8,     # V3 NEW: feature sampling per tree
}
v3_model=xgb.XGBClassifier(**model_parms)
v3_model.fit(data_x_training_scaled,data_y_training,
             eval_set=[(data_x_testing_scaled,data_y_testing)],
            verbose=50
)
print(f"\n✅ Training complete. Best iteration: {v3_model.best_iteration}")

Class imbalance ratio (fraud_weight): 582.41
(Each fraud sample treated as if it were this many normal samples)

[0]	validation_0-aucpr:0.55156
[50]	validation_0-aucpr:0.72204
[100]	validation_0-aucpr:0.80192
[150]	validation_0-aucpr:0.80714
[200]	validation_0-aucpr:0.81645
[250]	validation_0-aucpr:0.82292
[300]	validation_0-aucpr:0.83067
[350]	validation_0-aucpr:0.83690
[400]	validation_0-aucpr:0.83929
[450]	validation_0-aucpr:0.83926
[476]	validation_0-aucpr:0.83948

✅ Training complete. Best iteration: 426


In [41]:
#FINAL PREDICTIONS AND METRICS
#RAW PROBABILITY USED FOR FRONTEND THRESHOLD BAR
test_prob=v3_model.predict_proba(data_x_testing_scaled)[:, 1]
#RAW PREDICITON SCORES
test_pred=v3_model.predict(data_x_testing)
#CALUCLATING METRICS
tn,fp,fn,tp=confusion_matrix(data_y_testing,test_pred).ravel()
auprc=average_precision_score(data_y_testing, test_prob)

print("=" * 50)
print("CONFUSION MATRIX (At default 0.5 Threshold)")
print(f"  True Positives  (caught fraud):     {tp}")
print(f"  False Positives (innocent blocked): {fp}")
print(f"  False Negatives (missed fraud):     {fn}")
print(f"  True Negatives  (correct clear):    {tn}")
print("=" * 50)
print(f"  AUPRC Score: {auprc:.4f}")
print(f"  Recall (fraud caught): {tp/(tp+fn)*100:.2f}%")
print("=" * 50)
print()
print(classification_report(data_y_testing, test_pred, target_names=['Legit', 'Fraud']))


CONFUSION MATRIX (At default 0.5 Threshold)
  True Positives  (caught fraud):     74
  False Positives (innocent blocked): 19
  False Negatives (missed fraud):     21
  True Negatives  (correct clear):    55019
  AUPRC Score: 0.8404
  Recall (fraud caught): 77.89%

              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     55038
       Fraud       0.80      0.78      0.79        95

    accuracy                           1.00     55133
   macro avg       0.90      0.89      0.89     55133
weighted avg       1.00      1.00      1.00     55133



In [42]:
#NET ROI
annoyance_cost_per_fp=5 #cost of innocent customer blocked
amount_saved=amount_test[(data_y_testing.values==1) & (test_pred==1)].sum()
amount_missed=amount_test[(data_y_testing.values==1) & (test_pred==0)].sum()
false_Alarm_Cost= fp * annoyance_cost_per_fp
net_roi=amount_saved - false_Alarm_Cost
print("💰 BUSINESS IMPACT REPORT — V3 MODEL")
print("=" * 50)
print(f"  Fraud CAUGHT (saved):      ${amount_saved:>12,.2f}")
print(f"  Fraud MISSED (lost):       ${amount_missed:>12,.2f}")
print(f"  False alarm cost:          ${false_Alarm_Cost:>12,.2f}  ({fp} false alarms × ${annoyance_cost_per_fp})")
print(f"  NET ROI:                   ${net_roi:>12,.2f}")
print("=" * 50)
recall_pct = tp/(tp+fn)*100
print(f"  Fraud recall: {recall_pct:.1f}% of all fraud caught")

💰 BUSINESS IMPACT REPORT — V3 MODEL
  Fraud CAUGHT (saved):      $   10,185.73
  Fraud MISSED (lost):       $    3,395.65
  False alarm cost:          $       95.00  (19 false alarms × $5)
  NET ROI:                   $   10,090.73
  Fraud recall: 77.9% of all fraud caught


In [43]:
#MLOPS  GUARDRAILS

# Guardrails put conditions for monitoring model health and finding champion model
GUARDRAILS = {
    'min_auprc': 0.75,           # AUPRC must be at least this
    'max_fp_rate': 0.005,        # FP rate must be below 0.5% of legit transactions
    'min_recall': 0.70,          # Must catch at least 70% of fraud
}

fp_rate = fp / (fp + tn)
recall = tp / (tp + fn)

violations = []
if auprc < GUARDRAILS['min_auprc']:
    violations.append(f"⚠️  AUPRC {auprc:.4f} < required {GUARDRAILS['min_auprc']}")
if fp_rate > GUARDRAILS['max_fp_rate']:
    violations.append(f"⚠️  FP Rate {fp_rate:.5f} > allowed {GUARDRAILS['max_fp_rate']}")
if recall < GUARDRAILS['min_recall']:
    violations.append(f"⚠️  Recall {recall:.4f} < required {GUARDRAILS['min_recall']}")

health_status = "HEALTHY" if not violations else "DEGRADED"

print(f"🏥 MODEL HEALTH: {health_status}")
if violations:
    for v in violations:
        print(v)
else:
    print("   All guardrails passed ✅")



🏥 MODEL HEALTH: HEALTHY
   All guardrails passed ✅


In [44]:
# ── CELL 13: MLOps — Experiment Tracker ──────────────────────────────────

# V3 FIX: reads params from model dynamically instead of hardcoding
params = v3_model.get_params()

run_data = {
    "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "Version": "XGBoost_V3.0",
    # Hyperparams read from model directly — no hardcoding
    "n_estimators": v3_model.best_iteration,  # actual best, not max, using param[n_estimators would give us the max estimators like 600 or 300 best_iteration gives us the best number of trees used like 426]
    "max_depth": params['max_depth'],
    "learning_rate": params['learning_rate'],
    "subsample": params['subsample'],
    "colsample_bytree": params['colsample_bytree'],
    # Threshold
    "Threshold": 0.5,
    # Metrics
    "AUPRC": round(auprc, 4),
    "Recall": round(recall, 4),
    "FP_Rate": round(fp_rate, 6),
    "True_Positives": int(tp),
    "False_Positives": int(fp),
    "False_Negatives": int(fn),
    # Business
    "Amount_Saved_USD": round(amount_saved, 2),
    "Net_ROI_USD": round(net_roi, 2),
    # Health
    "Health_Status": health_status
}

LOG_FILE = "experiment_tracking_v3.csv"
tracker_df = pd.DataFrame([run_data])

if not os.path.exists(LOG_FILE):
    tracker_df.to_csv(LOG_FILE, index=False)
else:
    tracker_df.to_csv(LOG_FILE, mode='a', header=False, index=False)

print(f"📋 Run logged to {LOG_FILE}")
print("\nFull experiment log so far:")
print(pd.read_csv(LOG_FILE).to_string())

📋 Run logged to experiment_tracking_v3.csv

Full experiment log so far:
             Timestamp       Version  n_estimators  max_depth  learning_rate  subsample  colsample_bytree  Threshold   AUPRC  Recall   FP_Rate  True_Positives  False_Positives  False_Negatives  Amount_Saved_USD  Net_ROI_USD Health_Status
0  2026-04-21 17:25:20  XGBoost_V3.0           426          6           0.05        0.8               0.8        0.5  0.8404  0.7789  0.000345              74               19               21          10185.73     10090.73       HEALTHY


In [45]:
# ── CELL 14: MLOps — Model Registry (Champion/Challenger) ─────────────────

REGISTRY_FILE = "model_registry.json"
MODELS_DIR = "registered_models"
os.makedirs(MODELS_DIR, exist_ok=True)

current_best_auprc = 0.0
if os.path.exists(REGISTRY_FILE):
    with open(REGISTRY_FILE, 'r') as f:
        registry = json.load(f)
    current_best_auprc = registry.get('best_auprc', 0.0)
    print(f"📚 Current champion AUPRC: {current_best_auprc:.4f} ({registry.get('champion_version', 'none')})")
else:
    print("📚 No registry found — this will be the first registered model")

if health_status == "HEALTHY" and auprc > current_best_auprc:
    # Save model artefacts
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    model_path  = f"{MODELS_DIR}/v3_xgb_{timestamp}.joblib"
    scaler_path = f"{MODELS_DIR}/v3_scaler_{timestamp}.joblib"

    joblib.dump(v3_model, model_path)
    joblib.dump(scaler, scaler_path)

    # Update registry
    new_registry = {
        "champion_version": f"V3_{timestamp}",
        "best_auprc": round(auprc, 4),
        "threshold": 0.5, # FIXED: Hardcoded to 0.5 for the backend
        "model_path": model_path,
        "scaler_path": scaler_path,
        "registered_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "net_roi": round(net_roi, 2)
    }
    with open(REGISTRY_FILE, 'w') as f:
        json.dump(new_registry, f, indent=2)

    print(f"\n🏆 NEW CHAMPION REGISTERED!")
    print(f"   AUPRC: {auprc:.4f} (improved from {current_best_auprc:.4f})")
    print(f"   Model saved: {model_path}")
    print(f"   Threshold: 0.5 (Frontend dynamic)")
elif health_status != "HEALTHY":
    print("\n🚫 Model NOT registered — failed guardrail checks")
    print("   Previous champion remains active")
else:
    print(f"\n📊 Model NOT registered — AUPRC {auprc:.4f} did not beat champion {current_best_auprc:.4f}")
    print("   Previous champion remains active")

📚 Current champion AUPRC: 0.8427 (V3_20260421_0024)

📊 Model NOT registered — AUPRC 0.8404 did not beat champion 0.8427
   Previous champion remains active


In [46]:
# ── CELL 15: MLOps — Historical Backfill (One-Time Execution) ─────────────


# ==============================================================================
# ⚠️ ONE-TIME EXECUTION BLOCK: MLOps Historical Backfill
# ==============================================================================
# WHAT WE DID HERE:
# We used this script to crack open our old V1 (Random Forest) and V2 (Baseline XGBoost)
# .joblib files to automatically extract their exact hyperparameters. 
#
# WHY WE DID IT:
# To manually inject our past performance metrics (like the 68% Precision / 80% Recall 
# from V2) into our new V3 `experiment_tracking.csv`. This allowed us to build a 
# complete, chronological audit trail of our model's evolution without having to 
# rewrite the old data pipelines or retrain the old models.
#
# THE RULE:
# Keep this entire block commented out (Ctrl + /) now that the CSV is populated! 
# If you run it again, it will blindly paste duplicate copies of V1 and V2 into the log.
# ==============================================================================



In [ ]:
# # ── CELL 15: MLOps — Historical Backfill (for baseline ver) ─────────────
# #import joblib
# #import pandas as pd
# #import os
# #from datetime import datetime

# # ── 1. Load the Preserved Engine ──
# # Change this path to wherever your V2 model is saved
# baseline_model = joblib.load(r"C:\Users\vishw\Desktop\projects\mini projects aiml\FRUAD_DETECTION_V2\v2_xgboost_baseline_model.joblib") 

# # Automatically extract the exact parameters it was trained with!
# params = baseline_model.get_params()

# # ── 2. Build the Tracker Payload ──
# run_data = {
#     "Timestamp": "2026-04-15 14:00:00",  # Fake past date so it sits chronologically in your log
#     "Version": "XGBoost_V2.0_Baseline",
    
#     # 🧠 DYNAMIC EXTRACTION: Pulling straight from the joblib file
#     "n_estimators": params.get('n_estimators', 'N/A'), 
#     "max_depth": params.get('max_depth', 'N/A'),
#     "learning_rate": params.get('learning_rate', 'N/A'),
#     "subsample": params.get('subsample', 'N/A'),
#     "colsample_bytree": params.get('colsample_bytree', 'N/A'),
    
#     "Threshold": 0.5,
    
#     # 📊 MANUAL METRICS: Plug in the numbers from your old V2 printouts
#     "AUPRC": 0.8044,                     
#     "Recall": 0.8000,                    
#     "FP_Rate": 0.000818,                 
#     "True_Positives": 76,                
#     "False_Positives": 45,               
#     "False_Negatives": 19,               
#     "Amount_Saved_USD": 10187.00,         
#     "Net_ROI_USD": 9962.00,              
#     "Health_Status": "HEALTHY"
# }

# LOG_FILE = "experiment_tracking_v3.csv"
# hist_df = pd.DataFrame([run_data])

# # ── 3. Inject into the Ledger ──
# if os.path.exists(LOG_FILE):
#     hist_df.to_csv(LOG_FILE, mode='a', header=False, index=False)
#     print("✅ Joblib model successfully extracted and backfilled into the log!")
    
#     # Print the sorted log
#     full_log = pd.read_csv(LOG_FILE)
#     full_log = full_log.sort_values(by="Timestamp") 
#     print("\nFull Chronological Experiment Log:")
#     print(full_log.to_string())
# else:
#     print("⚠️ Error: experiment_tracking.csv not found.")

✅ Joblib model successfully extracted and backfilled into the log!

Full Chronological Experiment Log:
             Timestamp                Version  n_estimators  max_depth  learning_rate  subsample  colsample_bytree  Threshold   AUPRC  Recall   FP_Rate  True_Positives  False_Positives  False_Negatives  Amount_Saved_USD  Net_ROI_USD Health_Status
1  2026-04-15 14:00:00  XGBoost_V2.0_Baseline           200          5           0.05        NaN               NaN        0.5  0.8044  0.8000  0.000818              76               45               19          10187.00      9962.00       HEALTHY
0  2026-04-21 17:25:20           XGBoost_V3.0           426          6           0.05        0.8               0.8        0.5  0.8404  0.7789  0.000345              74               19               21          10185.73     10090.73       HEALTHY


In [ ]:
# # ── CELL 15: MLOps — Historical Backfill (for v1 ver) ─────────────
# import joblib
# import pandas as pd
# import os
# from datetime import datetime

# # ── 1. Load the Preserved Engine ──
# # Change this path to wherever your V2 model is saved
# v1_model = joblib.load(r"C:\Users\vishw\Desktop\projects\mini projects aiml\fraud_Detection_V1\fraud_model.joblib") 

# # Automatically extract the exact parameters it was trained with!
# params = v1_model.get_params()

# # ── 2. Build the Tracker Payload ──
# run_data = {
#     "Timestamp": "2026-04-15 14:00:00",  # Fake past date so it sits chronologically in your log
#     "Version": "random_forest",
    
#     # 🧠 DYNAMIC EXTRACTION: Pulling straight from the joblib file
#     "n_estimators": params.get('n_estimators', 'N/A'), 
#     "max_depth": params.get('max_depth', 'N/A'),
#     "learning_rate": params.get('learning_rate', 'N/A'),
#     "subsample": params.get('subsample', 'N/A'),
#     "colsample_bytree": params.get('colsample_bytree', 'N/A'),
    
#     "Threshold": 0.5,
    
#     # 📊 MANUAL METRICS: Plug in the numbers from your old V2 printouts
#     "AUPRC": 0.8016,                     
#     "Recall": 0.7474,                    
#     "FP_Rate": 0.00002,                 
#     "True_Positives": 71,                
#     "False_Positives": 1,               
#     "False_Negatives": 24,               
#     "Amount_Saved_USD": 10714.00,         
#     "Net_ROI_USD": 10709.00,              
#     "Health_Status": "HEALTHY"
# }

# LOG_FILE = "experiment_tracking_v3.csv"
# hist_df = pd.DataFrame([run_data])

# # ── 3. Inject into the Ledger ──
# if os.path.exists(LOG_FILE):
#     hist_df.to_csv(LOG_FILE, mode='a', header=False, index=False)
#     print("✅ Joblib model successfully extracted and backfilled into the log!")
    
#     # Print the sorted log
#     full_log = pd.read_csv(LOG_FILE)
#     full_log = full_log.sort_values(by="Timestamp") 
#     print("\nFull Chronological Experiment Log:")
#     print(full_log.to_string())
# else:
#     print("⚠️ Error: experiment_tracking.csv not found.")

✅ Joblib model successfully extracted and backfilled into the log!

Full Chronological Experiment Log:
             Timestamp                Version  n_estimators  max_depth  learning_rate  subsample  colsample_bytree  Threshold   AUPRC  Recall   FP_Rate  True_Positives  False_Positives  False_Negatives  Amount_Saved_USD  Net_ROI_USD Health_Status
1  2026-04-15 14:00:00  XGBoost_V2.0_Baseline           200          5           0.05        NaN               NaN        0.5  0.8044  0.8000  0.000818              76               45               19          10187.00      9962.00       HEALTHY
2  2026-04-15 14:00:00          random_forest           100         20            NaN        NaN               NaN        0.5  0.8016  0.7474  0.000020              71                1               24          10714.00     10709.00       HEALTHY
0  2026-04-21 17:25:20           XGBoost_V3.0           426          6           0.05        0.8               0.8        0.5  0.8404  0.7789  0.000345     

In [53]:
# ── CELL 15: MLOps — Load & Serve Champion Model ──────────────────────────

# This simulates what a serving layer does: it loads the registered champion
# and uses it for inference — NOT whatever model happens to be in memory

def load_champion_and_predict(new_transactions_df):
    """
    Production-style inference using the registered champion model.
    new_transactions_df: raw transaction data (same format as original CSV)
    """
    if not os.path.exists(REGISTRY_FILE):
        raise FileNotFoundError("No champion model registered yet!")

    with open(REGISTRY_FILE, 'r') as f:
        registry = json.load(f)

    model  = joblib.load(registry['model_path'])
    scaler = joblib.load(registry['scaler_path'])
   

    # Apply same feature engineering pipeline
    df = new_transactions_df.copy()
    df['Hour_of_Day'] = (df['Time'] // 3600) % 24
    df['IS_NIGHT'] = df['Hour_of_Day'].apply(lambda x: 1 if x <= 6 or x >= 23 else 0)
    df['LOG_AMOUNT'] = np.log1p(df['Amount'])
    df['AMOUNT_TIER'] = pd.cut(
        df['Amount'],
        bins=[-1, 1, 10, 100, 1000, float('inf')],
        labels=[0, 1, 2, 3, 4]
    ).astype(int)
    df.drop(['Time', 'Amount'], axis=1, inplace=True)
    if 'Class' in df.columns:
        df.drop('Class', axis=1, inplace=True)

    scaled = scaler.transform(df)
    probs = model.predict_proba(scaled)[:, 1]
    preds = (probs >= 0.5).astype(int)

    result = pd.DataFrame({'fraud_probability': probs, 'is_fraud': preds})
    return result


# Quick smoke test on 5 rows from original test set
raw_data_test = pd.read_csv(fraud_Data_path)
subset_test = [c for c in raw_data_test.columns if c not in ['Time', 'Class']]
raw_data_test.drop_duplicates(subset=subset_test, inplace=True)
sample_transactions = raw_data_test.head(5)

result = load_champion_and_predict(sample_transactions)
print("🔍 Champion model inference on 5 sample transactions:")
print(result)

🔍 Champion model inference on 5 sample transactions:
   fraud_probability  is_fraud
0       4.365420e-05         0
1       3.613559e-05         0
2       6.198006e-05         0
3       7.707096e-07         0
4       6.186378e-06         0
